# Setup and correctness

Reads the prepared manifest and executes a small real inverse test.

In [1]:
from pathlib import Path
import json, csv, sys
ROOT = Path.cwd()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
assert (ROOT / 'src' / 'revllm').exists(), 'Execute with repository root as working directory'
sys.path.insert(0, str(ROOT / 'src'))
from revllm.data import digest_file
def read_final(path):
    result = json.loads(path.read_text())
    updates = [json.loads(line) for line in (path.parent/'metrics.jsonl').read_text().splitlines()]
    assert sum(row['valid_targets'] for row in updates if row['status']=='COMMITTED') == result['committed_targets']
    return result


In [2]:
from revllm.model import ModelConfig, TinyGPT, count_unique_parameters
from revllm.correctness import reconstruction_probe
print({kind: count_unique_parameters(TinyGPT(ModelConfig(integrator=kind))) for kind in ('conventional','midpoint','coupled_euler','blended_midpoint')})

{'conventional': 19969152, 'midpoint': 19969152, 'coupled_euler': 19969152, 'blended_midpoint': 19969152}


In [3]:
model = TinyGPT(ModelConfig(vocab_size=32, block_size=8, n_layer=4, n_head=2, n_embd=16, integrator='midpoint', backward_mode='reconstructed')).double()
print(reconstruction_probe(model))

{'relative_reconstruction_error': 7.01748098244164e-18, 'threshold': 1e-09, 'status': 'PASS'}


In [4]:
manifest = json.loads((ROOT/'data'/'manifest.json').read_text())
print({'revision': manifest['dataset_revision'], 'tokenizer_sha256': manifest['tokenizer_sha256'], 'targets': {k:v['target_count'] for k,v in manifest['tapes'].items()}})

{'revision': 'f54c09fd23315a6f9c86f9dc80f725de7d8f9c64', 'tokenizer_sha256': '4764060791e54b50d8d51f4369684cf69205230a9334d08574826b18856b1d74', 'targets': {'dev': 262144, 'holdout': 1000000, 'train': 50000000}}
